In [1]:
import sys, os
ROOT = "/Users/fserracrespi/Desktop/PD_PROJECT_UOFL" 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from utils.Decode import decoder_DF
from utils.dataframe_cols import cols_asignacion
from utils.Prog_df import check_progression
from utils.Prog_df import progression_csv_upgrade
from utils.Prog_df import check_progression_multi
from utils.Prog_df import progression_multi_csv_upgrade
from utils.Prog_df import visit_csv_upgrade
from utils.Prog_df import check_visits
import json
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kurtosis, skew
import re
from datetime import datetime
import math
from itertools import combinations,  count
from mlxtend.frequent_patterns import apriori

In [2]:
BL_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_BL_12SEP2025.csv')
BL_data=BL_data.loc[BL_data['UPDRS_BL']==True,:]
print(f'Data BL visit Raw shape:{ BL_data.shape}')
BL_data.set_index('PATNO',inplace=True)
BL_data = BL_data.loc[:, BL_data.any(axis=0)]
BL_data = BL_data.loc[BL_data.any(axis=1), :]
print(f'Data BL visit no false col nor rows shape:{ BL_data.shape}')

V04_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_V04_12SEP2025.csv')
V04_data=V04_data.loc[V04_data['UPDRS_V04']==True,:]
print(f'Data V04 visit Raw shape:{ V04_data.shape}')
V04_data.set_index('PATNO',inplace=True)
V04_data=V04_data.loc[:, V04_data.any(axis=0)]
V04_data = V04_data.loc[V04_data.any(axis=1), :]
print(f'Data V04 visit no false col nor row shape:{ V04_data.shape}')


V06_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_V06_12SEP2025.csv')
V06_data=V06_data.loc[V06_data['UPDRS_V06']==True,:]
print(f'Data V06 visit Raw shape:{ V06_data.shape}')
V06_data.set_index('PATNO',inplace=True)
V06_data=V06_data.loc[:, V06_data.any(axis=0)]
V06_data = V06_data.loc[V06_data.any(axis=1), :]
print(f'Data V06 visit no false col nor row shape:{ V06_data.shape}')

V08_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_V08_12SEP2025.csv')
V08_data=V08_data.loc[V08_data['UPDRS_V08']==True,:]
print(f'Data V08 visit Raw shape:{ V08_data.shape}')
V08_data.set_index('PATNO',inplace=True)
V08_data=V08_data.loc[:, V08_data.any(axis=0)]
V08_data = V08_data.loc[V08_data.any(axis=1), :]
print(f'Data V08 visit no false col nor row shape:{ V08_data.shape}')

V10_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_V10_12SEP2025.csv')
V10_data=V10_data.loc[V10_data['UPDRS_V10']==True,:]
print(f'Data V10 visit Raw shape:{ V10_data.shape}')
V10_data.set_index('PATNO',inplace=True)
V10_data=V10_data.loc[:, V10_data.any(axis=0)]
V10_data = V10_data.loc[V10_data.any(axis=1), :]
print(f'Data V10 visit no false col nor row shape:{ V10_data.shape}')

V12_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES/VISITS_SUBJECT_V12_12SEP2025.csv')
V12_data=V12_data.loc[V12_data['UPDRS_V12']==True,:]
print(f'Data V12 visit Raw shape:{ V12_data.shape}')
V12_data.set_index('PATNO',inplace=True)
V12_data=V12_data.loc[:, V12_data.any(axis=0)]
V12_data = V12_data.loc[V12_data.any(axis=1), :]
print(f'Data V12 visit no false col nor row shape:{ V12_data.shape}')

Data BL visit Raw shape:(117, 27)
Data BL visit no false col nor rows shape:(117, 24)
Data V04 visit Raw shape:(491, 27)
Data V04 visit no false col nor row shape:(491, 25)
Data V06 visit Raw shape:(479, 27)
Data V06 visit no false col nor row shape:(479, 25)
Data V08 visit Raw shape:(378, 27)
Data V08 visit no false col nor row shape:(378, 25)
Data V10 visit Raw shape:(297, 27)
Data V10 visit no false col nor row shape:(297, 25)
Data V12 visit Raw shape:(249, 27)
Data V12 visit no false col nor row shape:(249, 24)


# BL Analysis

In [3]:
print(f'Data BL rows with all values True: {BL_data[BL_data.all(axis=1)].shape}')

Data BL rows with all values True: (0, 24)


In [4]:
def selector_apriori(df, min_support=0.01, top_n=5):
    """
    Encuentra los subconjuntos de columnas (itemsets) que maximizan
    el número de filas con todo True usando el algoritmo Apriori.

    Criterio:
      - Prioriza mayor número de filas (n_rows_true)
      - En caso de empate, el subconjunto con más columnas (itemset más largo)
    """
    print("🔍 Ejecutando Apriori...")
    result = apriori(df, min_support=min_support, use_colnames=True)

    if result.empty:
        print("⚠️ No se encontraron combinaciones con ese soporte mínimo.")
        return pd.DataFrame()

    # Calcular número de filas y longitud del itemset
    result["n_rows_true"] = (result["support"] * len(df)).astype(int)
    result["n_items"] = result["itemsets"].apply(len)

    # Ordenar: primero por filas True, luego por longitud del subset
    result = result.sort_values(["n_rows_true", "n_items"], ascending=[False, False])

    # Mostrar los mejores resultados
    print(f"\n✅ Top {top_n} combinaciones más frecuentes:")
    for i, row in result.head(top_n).iterrows():
        print(f"🔢 {i+1}. {set(row['itemsets'])} → {row['n_rows_true']} filas True ({row['n_items']} columnas)")

    # Elegir el mejor según los criterios
    best = result.iloc[0]
    print("\n🏆 Mejor subconjunto:", set(best["itemsets"]))
    print("📊 Filas con todo True:", best["n_rows_true"])
    print("🧩 Columnas en el subset:", best["n_items"])

    return result.head(top_n)


def selector_apriori_2(df, min_support=0.01, top_n=5):
    """
    Encuentra los subconjuntos de columnas (itemsets) más representativos
    usando el algoritmo Apriori.

    Criterio:
      1. Prioriza el número de columnas (más ítems)
      2. En caso de empate, el subconjunto con más filas True
    """
    print("🔍 Ejecutando Apriori...")
    result = apriori(df, min_support=min_support, use_colnames=True)

    if result.empty:
        print("⚠️ No se encontraron combinaciones con ese soporte mínimo.")
        return pd.DataFrame()

    # Calcular métricas adicionales
    result["n_rows_true"] = (result["support"] * len(df)).astype(int)
    result["n_items"] = result["itemsets"].apply(len)

    # Ordenar: primero por número de ítems, luego por filas True
    result = result.sort_values(["n_items", "n_rows_true"], ascending=[False, False])

    # Mostrar los mejores resultados
    print(f"\n✅ Top {top_n} combinaciones más frecuentes:")
    for i, row in result.head(top_n).iterrows():
        print(f"🔢 {i+1}. {set(row['itemsets'])} → {row['n_rows_true']} filas True ({row['n_items']} columnas)")

    # Elegir el mejor según el nuevo criterio
    best = result.iloc[0]
    print("\n🏆 Mejor subconjunto:", set(best["itemsets"]))
    print("📊 Filas con todo True:", best["n_rows_true"])
    print("🧩 Columnas en el subset:", best["n_items"])

    return result.head(top_n)



# V04 Analyisis

In [5]:
print(f'Data V04 rows with all values True: {V04_data[V04_data.all(axis=1)].shape}')

Data V04 rows with all values True: (0, 25)


In [6]:
# V04_Selector = selector_apriori(V04_data, min_support=0.01, top_n=10)
''''
🏆 Mejor subconjunto: {'NEURO_V04', 'UPDRS_V04'}
📊 Filas con todo True: 491
🧩 Columnas en el subset: 2
'''

"'\n🏆 Mejor subconjunto: {'NEURO_V04', 'UPDRS_V04'}\n📊 Filas con todo True: 491\n🧩 Columnas en el subset: 2\n"

In [7]:
V04_Selector = selector_apriori_2(V04_data, min_support=0.90, top_n=10)
'''''
🏆 Mejor subconjunto: {'EPWORTH_V04', 'SchwabADL_V04', 'AE_NO_CLINIC_V04', 'VITAL_V04', 'LETTER_V04', 'AE_CLINIC_V04', 'SUBJECT_V04', 'BENTONJLO_V04', 'SYMBOLDIGIT_V04', 'ANXIETY_V04', 'LEDD_MEDS_V04', 'MOCA_V04', 'HOPKINS_V04', 'NEURO_V04', 'NOT_PD_MEDS_V04', 'GERIATRICDEPRESSION_V04', 'UPDRS_V04', 'REMSLEEP_V04', 'SEMANTIC_V04'}
📊 Filas con todo True: 457
🧩 Columnas en el subset: 19
'''

🔍 Ejecutando Apriori...

✅ Top 10 combinaciones más frecuentes:
🔢 1060863. {'UPDRS_V04', 'SUBJECT_V04', 'GERIATRICDEPRESSION_V04', 'VITAL_V04', 'SEMANTIC_V04', 'REMSLEEP_V04', 'LETTER_V04', 'FeaturesPD_V04', 'LEDD_MEDS_V04', 'EPWORTH_V04', 'AE_NO_CLINIC_V04', 'NOT_PD_MEDS_V04', 'MOCA_V04', 'HOPKINS_V04', 'AE_CLINIC_V04', 'SchwabADL_V04', 'NEURO_V04', 'ANXIETY_V04', 'SYMBOLDIGIT_V04', 'BENTONJLO_V04'} → 443 filas True (20 columnas)
🔢 1060850. {'UPDRS_V04', 'SUBJECT_V04', 'GERIATRICDEPRESSION_V04', 'VITAL_V04', 'SEMANTIC_V04', 'REMSLEEP_V04', 'LETTER_V04', 'LEDD_MEDS_V04', 'EPWORTH_V04', 'AE_NO_CLINIC_V04', 'NOT_PD_MEDS_V04', 'MOCA_V04', 'HOPKINS_V04', 'AE_CLINIC_V04', 'SchwabADL_V04', 'NEURO_V04', 'ANXIETY_V04', 'SYMBOLDIGIT_V04', 'BENTONJLO_V04'} → 461 filas True (19 columnas)
🔢 1060859. {'UPDRS_V04', 'SUBJECT_V04', 'GERIATRICDEPRESSION_V04', 'VITAL_V04', 'SEMANTIC_V04', 'LETTER_V04', 'FeaturesPD_V04', 'LEDD_MEDS_V04', 'EPWORTH_V04', 'AE_NO_CLINIC_V04', 'NOT_PD_MEDS_V04', 'MOCA_V04', '

"''\n🏆 Mejor subconjunto: {'EPWORTH_V04', 'SchwabADL_V04', 'AE_NO_CLINIC_V04', 'VITAL_V04', 'LETTER_V04', 'AE_CLINIC_V04', 'SUBJECT_V04', 'BENTONJLO_V04', 'SYMBOLDIGIT_V04', 'ANXIETY_V04', 'LEDD_MEDS_V04', 'MOCA_V04', 'HOPKINS_V04', 'NEURO_V04', 'NOT_PD_MEDS_V04', 'GERIATRICDEPRESSION_V04', 'UPDRS_V04', 'REMSLEEP_V04', 'SEMANTIC_V04'}\n📊 Filas con todo True: 457\n🧩 Columnas en el subset: 19\n"

# V06 Analyisis

In [8]:
print(f'Data V06 rows with all values True: {V06_data[V06_data.all(axis=1)].shape}')

Data V06 rows with all values True: (0, 25)


In [9]:
# V06_Selector = selector_apriori(V06_data, min_support=0.01, top_n=10)
'''''
🏆 Mejor subconjunto: {'UPDRS_V06'}
📊 Filas con todo True: 479
🧩 Columnas en el subset: 1
'''

"''\n🏆 Mejor subconjunto: {'UPDRS_V06'}\n📊 Filas con todo True: 479\n🧩 Columnas en el subset: 1\n"

In [10]:
V06_Selector = selector_apriori_2(V06_data, min_support=0.90, top_n=10)
'''''
🏆 Mejor subconjunto: {'AE_CLINIC_V06', 'MOCA_V06', 'SUBJECT_V06', 'VITAL_V06', 'ANXIETY_V06', 'SEMANTIC_V06', 'NEURO_V06', 'LEDD_MEDS_V06', 'SchwabADL_V06', 'HOPKINS_V06', 'SYMBOLDIGIT_V06', 'AE_NO_CLINIC_V06', 'NOT_PD_MEDS_V06', 'EPWORTH_V06', 'UPDRS_V06', 'GERIATRICDEPRESSION_V06', 'BENTONJLO_V06', 'LETTER_V06'}
📊 Filas con todo True: 438
🧩 Columnas en el subset: 18

'''

🔍 Ejecutando Apriori...

✅ Top 10 combinaciones más frecuentes:
🔢 1061246. {'SUBJECT_V06', 'REMSLEEP_V06', 'SYMBOLDIGIT_V06', 'GERIATRICDEPRESSION_V06', 'SchwabADL_V06', 'EPWORTH_V06', 'VITAL_V06', 'BENTONJLO_V06', 'NEURO_V06', 'MOCA_V06', 'ANXIETY_V06', 'LEDD_MEDS_V06', 'NOT_PD_MEDS_V06', 'AE_CLINIC_V06', 'LETTER_V06', 'SEMANTIC_V06', 'AE_NO_CLINIC_V06', 'UPDRS_V06', 'HOPKINS_V06'} → 439 filas True (19 columnas)
🔢 1061247. {'SUBJECT_V06', 'SYMBOLDIGIT_V06', 'GERIATRICDEPRESSION_V06', 'SchwabADL_V06', 'EPWORTH_V06', 'VITAL_V06', 'BENTONJLO_V06', 'NEURO_V06', 'MOCA_V06', 'ANXIETY_V06', 'LEDD_MEDS_V06', 'NOT_PD_MEDS_V06', 'AE_CLINIC_V06', 'LETTER_V06', 'SEMANTIC_V06', 'FeaturesPD_V06', 'AE_NO_CLINIC_V06', 'UPDRS_V06', 'HOPKINS_V06'} → 432 filas True (19 columnas)
🔢 1061229. {'SUBJECT_V06', 'SYMBOLDIGIT_V06', 'GERIATRICDEPRESSION_V06', 'SchwabADL_V06', 'EPWORTH_V06', 'VITAL_V06', 'BENTONJLO_V06', 'NEURO_V06', 'MOCA_V06', 'ANXIETY_V06', 'LEDD_MEDS_V06', 'NOT_PD_MEDS_V06', 'AE_CLINIC_V06', 

"''\n🏆 Mejor subconjunto: {'AE_CLINIC_V06', 'MOCA_V06', 'SUBJECT_V06', 'VITAL_V06', 'ANXIETY_V06', 'SEMANTIC_V06', 'NEURO_V06', 'LEDD_MEDS_V06', 'SchwabADL_V06', 'HOPKINS_V06', 'SYMBOLDIGIT_V06', 'AE_NO_CLINIC_V06', 'NOT_PD_MEDS_V06', 'EPWORTH_V06', 'UPDRS_V06', 'GERIATRICDEPRESSION_V06', 'BENTONJLO_V06', 'LETTER_V06'}\n📊 Filas con todo True: 438\n🧩 Columnas en el subset: 18\n\n"

# V08 Analyisis

In [11]:
print(f'Data V08 rows with all values True: {V08_data[V08_data.all(axis=1)].shape}')

Data V08 rows with all values True: (0, 25)


In [12]:
# V08_Selector = selector_apriori(V08_data, min_support=0.01, top_n=10)
'''
🏆 Mejor subconjunto: {'NEURO_V08', 'SchwabADL_V08', 'UPDRS_V08'}
📊 Filas con todo True: 378
🧩 Columnas en el subset: 3
'''


"\n🏆 Mejor subconjunto: {'NEURO_V08', 'SchwabADL_V08', 'UPDRS_V08'}\n📊 Filas con todo True: 378\n🧩 Columnas en el subset: 3\n"

In [13]:
V08_Selector = selector_apriori_2(V08_data, min_support=0.90, top_n=10)
'''''
🏆 Mejor subconjunto: {'AE_CLINIC_V08', 'EPWORTH_V08', 'SUBJECT_V08', 'VITAL_V08', 'SchwabADL_V08', 'GERIATRICDEPRESSION_V08', 'HOPKINS_V08', 'SEMANTIC_V08', 'REMSLEEP_V08', 'LEDD_MEDS_V08', 'NOT_PD_MEDS_V08', 'MOCA_V08', 'LETTER_V08', 'SYMBOLDIGIT_V08', 'FeaturesPD_V08', 'BENTONJLO_V08', 'AE_NO_CLINIC_V08', 'NEURO_V08', 'ANXIETY_V08', 'UPDRS_V08'}
📊 Filas con todo True: 346
🧩 Columnas en el subset: 20
'''

🔍 Ejecutando Apriori...

✅ Top 10 combinaciones más frecuentes:
🔢 1447167. {'MOCA_V08', 'LEDD_MEDS_V08', 'BENTONJLO_V08', 'SUBJECT_V08', 'SYMBOLDIGIT_V08', 'SchwabADL_V08', 'EPWORTH_V08', 'UPDRS_V08', 'FeaturesPD_V08', 'NEURO_V08', 'AE_CLINIC_V08', 'NOT_PD_MEDS_V08', 'ANXIETY_V08', 'HOPKINS_V08', 'AE_NO_CLINIC_V08', 'GERIATRICDEPRESSION_V08', 'REMSLEEP_V08', 'VITAL_V08', 'LETTER_V08', 'SEMANTIC_V08'} → 351 filas True (20 columnas)
🔢 1447163. {'MOCA_V08', 'LEDD_MEDS_V08', 'BENTONJLO_V08', 'SUBJECT_V08', 'SYMBOLDIGIT_V08', 'SchwabADL_V08', 'EPWORTH_V08', 'UPDRS_V08', 'FeaturesPD_V08', 'NEURO_V08', 'AE_CLINIC_V08', 'NOT_PD_MEDS_V08', 'ANXIETY_V08', 'HOPKINS_V08', 'AE_NO_CLINIC_V08', 'GERIATRICDEPRESSION_V08', 'VITAL_V08', 'LETTER_V08', 'SEMANTIC_V08'} → 356 filas True (19 columnas)
🔢 1447164. {'MOCA_V08', 'LEDD_MEDS_V08', 'BENTONJLO_V08', 'SUBJECT_V08', 'SYMBOLDIGIT_V08', 'SchwabADL_V08', 'UPDRS_V08', 'FeaturesPD_V08', 'NEURO_V08', 'AE_CLINIC_V08', 'NOT_PD_MEDS_V08', 'ANXIETY_V08', 'HOPKI

"''\n🏆 Mejor subconjunto: {'AE_CLINIC_V08', 'EPWORTH_V08', 'SUBJECT_V08', 'VITAL_V08', 'SchwabADL_V08', 'GERIATRICDEPRESSION_V08', 'HOPKINS_V08', 'SEMANTIC_V08', 'REMSLEEP_V08', 'LEDD_MEDS_V08', 'NOT_PD_MEDS_V08', 'MOCA_V08', 'LETTER_V08', 'SYMBOLDIGIT_V08', 'FeaturesPD_V08', 'BENTONJLO_V08', 'AE_NO_CLINIC_V08', 'NEURO_V08', 'ANXIETY_V08', 'UPDRS_V08'}\n📊 Filas con todo True: 346\n🧩 Columnas en el subset: 20\n"

# V10 Analyisis

In [14]:
print(f'Data V04 rows with all values True: {V10_data[V10_data.all(axis=1)].shape}')

Data V04 rows with all values True: (0, 25)


In [15]:
#V10_Selector = selector_apriori(V10_data, min_support=0.01, top_n=10)
'''''
🏆 Mejor subconjunto: {'UPDRS_V10', 'NEURO_V10', 'FeaturesPD_V10', 'SchwabADL_V10'}
📊 Filas con todo True: 297
🧩 Columnas en el subset: 4
'''


"''\n🏆 Mejor subconjunto: {'UPDRS_V10', 'NEURO_V10', 'FeaturesPD_V10', 'SchwabADL_V10'}\n📊 Filas con todo True: 297\n🧩 Columnas en el subset: 4\n"

In [16]:
V10_Selector = selector_apriori_2(V10_data, min_support=0.90, top_n=10)
''''
🏆 Mejor subconjunto: {'GERIATRICDEPRESSION_V10', 'HOPKINS_V10', 'SchwabADL_V10', 'FeaturesPD_V10', 'VITAL_V10', 'AE_NO_CLINIC_V10', 'NEURO_V10', 'MOCA_V10', 'LETTER_V10', 'AE_CLINIC_V10', 'SUBJECT_V10', 'UPDRS_V10', 'NOT_PD_MEDS_V10', 'EPWORTH_V10', 'SYMBOLDIGIT_V10', 'SEMANTIC_V10', 'ANXIETY_V10', 'BENTONJLO_V10', 'REMSLEEP_V10', 'LEDD_MEDS_V10'}
📊 Filas con todo True: 278
🧩 Columnas en el subset: 20
'''

🔍 Ejecutando Apriori...

✅ Top 10 combinaciones más frecuentes:
🔢 1253375. {'BENTONJLO_V10', 'EPWORTH_V10', 'SEMANTIC_V10', 'REMSLEEP_V10', 'FeaturesPD_V10', 'LETTER_V10', 'MOCA_V10', 'ANXIETY_V10', 'NEURO_V10', 'AE_NO_CLINIC_V10', 'LEDD_MEDS_V10', 'VITAL_V10', 'SchwabADL_V10', 'HOPKINS_V10', 'NOT_PD_MEDS_V10', 'SUBJECT_V10', 'UPDRS_V10', 'AE_CLINIC_V10', 'SYMBOLDIGIT_V10', 'GERIATRICDEPRESSION_V10'} → 278 filas True (20 columnas)
🔢 1253356. {'BENTONJLO_V10', 'EPWORTH_V10', 'SEMANTIC_V10', 'REMSLEEP_V10', 'FeaturesPD_V10', 'LETTER_V10', 'MOCA_V10', 'ANXIETY_V10', 'NEURO_V10', 'AE_NO_CLINIC_V10', 'VITAL_V10', 'SchwabADL_V10', 'HOPKINS_V10', 'NOT_PD_MEDS_V10', 'SUBJECT_V10', 'UPDRS_V10', 'AE_CLINIC_V10', 'SYMBOLDIGIT_V10', 'GERIATRICDEPRESSION_V10'} → 281 filas True (19 columnas)
🔢 1253371. {'BENTONJLO_V10', 'EPWORTH_V10', 'SEMANTIC_V10', 'FeaturesPD_V10', 'LETTER_V10', 'MOCA_V10', 'ANXIETY_V10', 'NEURO_V10', 'AE_NO_CLINIC_V10', 'LEDD_MEDS_V10', 'VITAL_V10', 'SchwabADL_V10', 'HOPKINS_V10

"'\n🏆 Mejor subconjunto: {'GERIATRICDEPRESSION_V10', 'HOPKINS_V10', 'SchwabADL_V10', 'FeaturesPD_V10', 'VITAL_V10', 'AE_NO_CLINIC_V10', 'NEURO_V10', 'MOCA_V10', 'LETTER_V10', 'AE_CLINIC_V10', 'SUBJECT_V10', 'UPDRS_V10', 'NOT_PD_MEDS_V10', 'EPWORTH_V10', 'SYMBOLDIGIT_V10', 'SEMANTIC_V10', 'ANXIETY_V10', 'BENTONJLO_V10', 'REMSLEEP_V10', 'LEDD_MEDS_V10'}\n📊 Filas con todo True: 278\n🧩 Columnas en el subset: 20\n"

# V12 Analyisis

In [17]:
print(f'Data V12 rows with all values True: {V12_data[V12_data.all(axis=1)].shape}')

Data V12 rows with all values True: (0, 24)


In [18]:
#V12_Selector = selector_apriori(V12_data, min_support=0.01, top_n=10)
'''''
🏆 Mejor subconjunto: {'AE_CLINIC_V12', 'AE_NO_CLINIC_V12', 'VITAL_V12', 'LEDD_MEDS_V12', 'SchwabADL_V12', 'ANXIETY_V12', 'NEURO_V12', 'NOT_PD_MEDS_V12', 'UPDRS_V12', 'SUBJECT_V12', 'GERIATRICDEPRESSION_V12'}
📊 Filas con todo True: 249
🧩 Columnas en el subset: 11
'''


"''\n🏆 Mejor subconjunto: {'AE_CLINIC_V12', 'AE_NO_CLINIC_V12', 'VITAL_V12', 'LEDD_MEDS_V12', 'SchwabADL_V12', 'ANXIETY_V12', 'NEURO_V12', 'NOT_PD_MEDS_V12', 'UPDRS_V12', 'SUBJECT_V12', 'GERIATRICDEPRESSION_V12'}\n📊 Filas con todo True: 249\n🧩 Columnas en el subset: 11\n"

In [19]:
V12_Selector = selector_apriori_2(V12_data, min_support=0.90, top_n=10)

'''''
🏆 Mejor subconjunto: {'LEDD_MEDS_V12', 'VITAL_V12', 'SchwabADL_V12', 'ANXIETY_V12', 'FeaturesPD_V12', 'HOPKINS_V12', 'LETTER_V12', 'NOT_PD_MEDS_V12', 'BENTONJLO_V12', 'SYMBOLDIGIT_V12', 'EPWORTH_V12', 'UPDRS_V12', 'GERIATRICDEPRESSION_V12', 'AE_CLINIC_V12', 'MOCA_V12', 'AE_NO_CLINIC_V12', 'NEURO_V12', 'REMSLEEP_V12', 'SEMANTIC_V12', 'SUBJECT_V12'}
📊 Filas con todo True: 237
🧩 Columnas en el subset: 20
'''

🔍 Ejecutando Apriori...

✅ Top 10 combinaciones más frecuentes:
🔢 1048575. {'UPDRS_V12', 'EPWORTH_V12', 'GERIATRICDEPRESSION_V12', 'AE_CLINIC_V12', 'ANXIETY_V12', 'MOCA_V12', 'VITAL_V12', 'SEMANTIC_V12', 'FeaturesPD_V12', 'SUBJECT_V12', 'AE_NO_CLINIC_V12', 'REMSLEEP_V12', 'LETTER_V12', 'LEDD_MEDS_V12', 'HOPKINS_V12', 'SYMBOLDIGIT_V12', 'BENTONJLO_V12', 'NOT_PD_MEDS_V12', 'SchwabADL_V12', 'NEURO_V12'} → 235 filas True (20 columnas)
🔢 1048571. {'UPDRS_V12', 'EPWORTH_V12', 'GERIATRICDEPRESSION_V12', 'AE_CLINIC_V12', 'ANXIETY_V12', 'MOCA_V12', 'VITAL_V12', 'SEMANTIC_V12', 'FeaturesPD_V12', 'SUBJECT_V12', 'AE_NO_CLINIC_V12', 'LETTER_V12', 'LEDD_MEDS_V12', 'HOPKINS_V12', 'SYMBOLDIGIT_V12', 'BENTONJLO_V12', 'NOT_PD_MEDS_V12', 'SchwabADL_V12', 'NEURO_V12'} → 238 filas True (19 columnas)
🔢 1048556. {'UPDRS_V12', 'EPWORTH_V12', 'GERIATRICDEPRESSION_V12', 'AE_CLINIC_V12', 'ANXIETY_V12', 'MOCA_V12', 'VITAL_V12', 'SEMANTIC_V12', 'FeaturesPD_V12', 'SUBJECT_V12', 'AE_NO_CLINIC_V12', 'REMSLEEP_V12', '

"''\n🏆 Mejor subconjunto: {'LEDD_MEDS_V12', 'VITAL_V12', 'SchwabADL_V12', 'ANXIETY_V12', 'FeaturesPD_V12', 'HOPKINS_V12', 'LETTER_V12', 'NOT_PD_MEDS_V12', 'BENTONJLO_V12', 'SYMBOLDIGIT_V12', 'EPWORTH_V12', 'UPDRS_V12', 'GERIATRICDEPRESSION_V12', 'AE_CLINIC_V12', 'MOCA_V12', 'AE_NO_CLINIC_V12', 'NEURO_V12', 'REMSLEEP_V12', 'SEMANTIC_V12', 'SUBJECT_V12'}\n📊 Filas con todo True: 237\n🧩 Columnas en el subset: 20\n"

# Union de datos 1Y

In [20]:

L_V04 = [re.sub('_V04$', '', item) for item in V04_Selector.iloc[0]['itemsets']]
L_V06 = [re.sub('_V06$', '', item) for item in V06_Selector.iloc[0]['itemsets']]
L_V08 = [re.sub('_V08$', '', item) for item in V08_Selector.iloc[0]['itemsets']]
L_V10 = [re.sub('_V10$', '', item) for item in V10_Selector.iloc[0]['itemsets']]
L_V12 = [re.sub('_V12$', '', item) for item in V12_Selector.iloc[0]['itemsets']]

# Intersección de todas las listas
interseccion =  set(L_V04) & set(L_V06) & set(L_V08) & set(L_V10) & set(L_V12)

common_files=list(interseccion)

In [21]:
print(f'Common files in all visits selected by Apriori:{common_files}, len:{len(common_files)}')

Common files in all visits selected by Apriori:['LETTER', 'LEDD_MEDS', 'NOT_PD_MEDS', 'ANXIETY', 'EPWORTH', 'UPDRS', 'VITAL', 'SEMANTIC', 'GERIATRICDEPRESSION', 'SYMBOLDIGIT', 'SUBJECT', 'AE_CLINIC', 'AE_NO_CLINIC', 'HOPKINS', 'MOCA', 'SchwabADL', 'REMSLEEP', 'NEURO', 'BENTONJLO'], len:19


In [22]:
# Reset index

V04_data.reset_index(inplace=True)
V06_data.reset_index(inplace=True)
V08_data.reset_index(inplace=True)
V10_data.reset_index(inplace=True)
V12_data.reset_index(inplace=True)


# ---- V04 ----
common_files_V04 = ['PATNO'] + [col + "_V04" for col in common_files]
V04_data = V04_data[common_files_V04]
V04_data = V04_data[V04_data.all(axis=1)]
print(f'V04_data shape after common cols selection: {V04_data.shape}')
V04_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS_V04_data.csv')


# ---- V06 ----
common_files_V06 = ['PATNO'] + [col + "_V06" for col in common_files]
V06_data = V06_data[common_files_V06]
V06_data = V06_data[V06_data.all(axis=1)]
print(f'V06_data shape after common cols selection: {V06_data.shape}')
V06_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS_V06_data.csv')


# ---- V08 ----
common_files_V08 = ['PATNO'] + [col + "_V08" for col in common_files]
V08_data = V08_data[common_files_V08]
V08_data = V08_data[V08_data.all(axis=1)]
print(f'V08_data shape after common cols selection: {V08_data.shape}')
V06_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS_V08_data.csv')


# ---- V10 ----
common_files_V10 = ['PATNO'] + [col + "_V10" for col in common_files]
V10_data = V10_data[common_files_V10]
V10_data = V10_data[V10_data.all(axis=1)]
print(f'V10_data shape after common cols selection: {V10_data.shape}')
V10_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS_V10_data.csv')


# ---- V12 ----
common_files_V12 = ['PATNO'] + [col + "_V12" for col in common_files]
V12_data = V12_data[common_files_V12]
V12_data = V12_data[V12_data.all(axis=1)]
print(f'V12_data shape after common cols selection: {V12_data.shape}')
V12_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS_V12_data.csv')


V04_data shape after common cols selection: (461, 20)
V06_data shape after common cols selection: (439, 20)
V08_data shape after common cols selection: (353, 20)
V10_data shape after common cols selection: (278, 20)
V12_data shape after common cols selection: (237, 20)


In [23]:

V04_V06_data=pd.merge(V04_data,V06_data[['PATNO','UPDRS_V06']], how='inner', on='PATNO')
V04_V06_data['Visit ID']='V04_V06'
print(f'V04_V06_data shape after merge: {V04_V06_data.shape}')
V04_V06_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS1Y_V04_V06_data.csv')

V06_V08_data=pd.merge(V06_data,V08_data[['PATNO','UPDRS_V08']], how='inner', on='PATNO')
V06_V08_data['Visit ID']='V06_V08'
print(f'V06_V08_data shape after merge: {V06_V08_data.shape}')
V06_V08_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS1Y_V06_V08_data.csv')

V08_V10_data=pd.merge(V08_data,V10_data[['PATNO','UPDRS_V10']], how='inner', on='PATNO')
V08_V10_data['Visit ID']='V08_V10'
print(f'V08_V10_data shape after merge: {V08_V10_data.shape}')
V08_V10_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS1Y_V08_V10_data.csv')

V10_V12_data=pd.merge(V10_data,V12_data[['PATNO','UPDRS_V12']], how='inner', on='PATNO')
V10_V12_data['Visit ID']='V10_V12'
print(f'V10_V12_data shape after merge: {V10_V12_data.shape}')
V10_V12_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS1Y_V10_V12_data.csv')



V04_V06_data shape after merge: (279, 22)
V06_V08_data shape after merge: (245, 22)
V08_V10_data shape after merge: (204, 22)
V10_V12_data shape after merge: (183, 22)


In [24]:
DATA_PATH_DICT = {'EPWORTH':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_EPWORTH_12SEP2025.csv',
                    'ANXIETY':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ANXIETY_12SEP2025.csv',
                    'LETTER':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_LETTER_12SEP2025.csv',
                    'GERIATRICDEPRESSION':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_GERIATRICDEPRESSION_12SEP2025.csv',
                    'NEURO':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_NEURO_12SEP2025.csv',
                    'HOPKINS':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_HOPKINS_12SEP2025.csv',
                    'VITAL':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_VITAL_12SEP2025.csv',
                    'LEDD_MEDS':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_LEDD_MEDS_12SEP2025.csv',
                    'UPDRS':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_UPDRS_12SEP2025.csv',
                    'SchwabADL':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SchwabADL_12SEP2025.csv',
                    'SEMANTIC':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SEMANTIC_12SEP2025.csv',
                    'MOCA':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_MoCA_12SEP2025.csv',
                    'SUBJECT':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SUBJECT_CHARACTERISTICS_12SEP2025.csv',
                    'SYMBOLDIGIT':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SYMBOLDIGIT_12SEP2025.csv',
                    'BENTONJLO':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_BENTONJLO_12SEP2025.csv',
                    'AE_CLINIC':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_AE_CLINIC_12SEP2025.csv',
                    'NOT_PD_MEDS':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_NOT_PD_MEDS_12SEP2025.csv',
                    'AE_NO_CLINIC':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_AE_NO_CLINIC_12SEP2025.csv',
                    'REMSLEEP':'/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_REMSLEEP_12SEP2025.csv'}

def data_loader(df,id1,id2):

    list_data = df.columns.tolist()
    list_id1 = [w.replace("_"+id1, "") for w in list_data if w.endswith("_"+id1)]
    list_id2 = [w.replace("_"+id2, "") for w in list_data if w.endswith("_"+id2)]

    dict_data={id1: list_id1, id2: list_id2}
    ids = df['PATNO'].astype(str).tolist()
    visit = [id1 + "_" + id2] * len(ids)
    new_df=pd.DataFrame({'PATNO': ids,'Visit ID': visit})
    for key in dict_data:
        if key==id1:
            for col in dict_data[key]:
                df_specific = pd.read_csv(DATA_PATH_DICT[col])
                df_specific['PATNO']=df_specific['PATNO'].astype(str)
                df_specific = df_specific.loc[(df_specific['PATNO'].isin(ids)) & (df_specific['Visit ID']==id1), :]
                df_specific.drop(columns=['Visit ID'], inplace=True)
                new_df = pd.merge(new_df, df_specific, how='left', on='PATNO')

        if key==id2:
            for col in dict_data[key]:
                df_specific = pd.read_csv(DATA_PATH_DICT[col])
                df_specific['PATNO']=df_specific['PATNO'].astype(str)
                df_specific = df_specific.loc[(df_specific['PATNO'].isin(ids)) & (df_specific['Visit ID']==id2), :]
                df_specific['MDS-UPDRS Total Score 1YearProg']=df_specific[['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score']].astype(float).sum(axis=1)
                df_specific['UPDRS_I_1YearProg']=df_specific['MDS-UPDRS Part I (Patient Questionnaire) Total Score']
                df_specific['UPDRS_II_1YearProg']=df_specific['MDS-UPDRS Part II Total Score']
                df_specific['UPDRS_III_1YearProg']=df_specific['MDS-UPDRS Part III Total Score']
                df_specific['UPDRS_IV_1YearProg']=df_specific['MDS-UPDRS Part IV Total Score']
                df_specific= df_specific[['PATNO','MDS-UPDRS Total Score 1YearProg','UPDRS_I_1YearProg','UPDRS_II_1YearProg','UPDRS_III_1YearProg','UPDRS_IV_1YearProg']]
                new_df = pd.merge(new_df, df_specific, how='left', on='PATNO')

    new_df['MDS-UPDRS Total Score']=new_df[['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score']].astype(float).sum(axis=1)
    new_df['Δ TOTAL_UPDRS_1Y']=new_df['MDS-UPDRS Total Score 1YearProg'] - new_df['MDS-UPDRS Total Score']
    new_df['Δ UPDRS_I_1Y']=new_df['UPDRS_I_1YearProg'] - new_df['MDS-UPDRS Part I (Patient Questionnaire) Total Score']
    new_df['Δ UPDRS_II_1Y']=new_df['UPDRS_II_1YearProg'] - new_df['MDS-UPDRS Part II Total Score']
    new_df['Δ UPDRS_III_1Y']=new_df['UPDRS_III_1YearProg'] - new_df['MDS-UPDRS Part III Total Score']
    new_df['Δ UPDRS_IV_1Y']=new_df['UPDRS_IV_1YearProg'] - new_df['MDS-UPDRS Part IV Total Score']

    return new_df


V04_V06_final=data_loader(V04_V06_data,'V04','V06')
print(f'V04_V06_final shape:{ V04_V06_final.shape}')
V04_V06_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS1Y_data_V04_V06_final.csv')

V06_V08_final=data_loader(V06_V08_data,'V06','V08')
print(f'V06_V08_final shape:{ V06_V08_final.shape}')
V06_V08_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS1Y_data_V06_V08_final.csv')

V08_V10_final=data_loader(V08_V10_data,'V08','V10')
print(f'V08_V10_final shape:{ V08_V10_final.shape}')
V08_V10_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS1Y_data_V08_V10_final.csv')

V10_V12_final=data_loader(V10_V12_data,'V10','V12')
print(f'V10_V12_final shape:{ V10_V12_final.shape}')
V10_V12_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS1Y_data_V10_V12_final.csv')

# data final concatenada
FINAL_DATA=pd.concat([V04_V06_final,V06_V08_final,V08_V10_final,V10_V12_final], axis=0)
print(f'FINAL_DATA shape:{ FINAL_DATA.shape}')
FINAL_DATA.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')


V04_V06_final shape:(279, 501)
V06_V08_final shape:(245, 501)
V08_V10_final shape:(204, 501)
V10_V12_final shape:(183, 501)
FINAL_DATA shape:(911, 501)


In [25]:
V04_PATNO_VISITs = V04_data[['PATNO']].copy()
V04_PATNO_VISITs['V04_Presence'] = True

V06_PATNO_VISITs = V06_data[['PATNO']].copy()
V06_PATNO_VISITs['V06_Presence'] = True

V08_PATNO_VISITs = V08_data[['PATNO']].copy()
V08_PATNO_VISITs['V08_Presence'] = True

V10_PATNO_VISITs = V10_data[['PATNO']].copy()
V10_PATNO_VISITs['V10_Presence'] = True

V12_PATNO_VISITs = V12_data[['PATNO']].copy()
V12_PATNO_VISITs['V12_Presence'] = True

# Merge all visits together on PATNO
visit_seq = V04_PATNO_VISITs
visit_seq_V04_V06 = pd.merge(visit_seq, V06_PATNO_VISITs, on='PATNO', how='inner')
print(f'V04-V06 merged shape: {visit_seq_V04_V06.shape}')

visit_seq_V04_V06_V08 = pd.merge(visit_seq_V04_V06, V08_PATNO_VISITs, on='PATNO', how='inner')
print(f'V04-V06-V08 merged shape: {visit_seq_V04_V06_V08.shape}')

visit_seq_V04_V06_V08_V10 = pd.merge(visit_seq_V04_V06_V08, V10_PATNO_VISITs, on='PATNO', how='inner')
print(f'V04-V06-V08-V10 merged shape: {visit_seq_V04_V06_V08_V10.shape}')

visit_seq_V04_V06_V08_V10_V12 = pd.merge(visit_seq_V04_V06_V08_V10, V12_PATNO_VISITs, on='PATNO', how='inner')
print(f'V04-V06-V08-V10-V12 merged shape: {visit_seq_V04_V06_V08_V10_V12.shape}')


# Final visit sequence DataFrame
visit_seq_V04_V06.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence Matrix Sequence/visit_seq_V04_V06.csv', index=False)
visit_seq_V04_V06_V08.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence Matrix Sequence/visit_seq_V04_V06_V08.csv', index=False)
visit_seq_V04_V06_V08_V10.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence Matrix Sequence/visit_seq_V04_V06_V08_V10.csv', index=False)
visit_seq_V04_V06_V08_V10_V12.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence Matrix Sequence/visit_seq_V04_V06_V08_V10_V12.csv', index=False)





V04-V06 merged shape: (279, 3)
V04-V06-V08 merged shape: (158, 4)
V04-V06-V08-V10 merged shape: (91, 5)
V04-V06-V08-V10-V12 merged shape: (58, 6)


# Subject Data Model

In [26]:
subject_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
subject_data.drop(columns=['Unnamed: 0'], inplace=True)
subject_data['PATNO']=subject_data['PATNO'].astype(str)
subject_data


# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation1=col_assignation.loc[col_assignation['df_main'].isin(['SUBJECT']),:]
col_assignation2=col_assignation.loc[col_assignation['df_main'].isin(['MEDICAL']) & col_assignation['df_secundario'].isin(['VITAL']),:]
col_assignation=col_assignation1['df_secundario_col'].tolist() + col_assignation2['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

subject_data = subject_data.loc[:, subject_data.columns.isin(col_assignation)]
subject_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Subject_1Year_12SEP2025.csv')
subject_data


,PATNO,Visit ID,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,Height (cm),Weight (kg),Temperature (Celsius),Supine BP - systolic (mmHg),...,AGE_AT_VISIT,DATE_VISIT,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,8,9.0,17,2.0,180.0,115.0,36.6,131.0,...,56.7,2021-12-01,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,2,7.0,23,0.0,172.0,68.0,36.3,128.0,...,68.4,2022-02-01,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,6,5.0,14,0.0,159.0,78.7,36.4,132.0,...,67.0,2022-01-01,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,7,17.0,17,3.0,167.0,59.2,36.5,108.0,...,71.0,2022-05-01,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,6,5.0,14,2.0,175.0,81.1,36.7,116.0,...,63.2,2022-03-01,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,14,12.0,20,3.0,178.0,100.3,36.9,124.0,...,60.6,2023-03-01,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,5,8.0,16,5.0,175.0,64.4,36.1,160.0,...,61.6,2022-10-01,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,16,12.0,32,3.0,160.0,69.9,36.7,155.0,...,68.3,2023-07-01,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,11,7.0,17,5.0,169.0,78.0,37.0,153.0,...,58.9,2023-02-01,44.0,27,40.0,4.0,-1,0.0,10,-5.0


In [27]:
subject_data.columns

Index(['PATNO', 'Visit ID',
       'MDS-UPDRS Part I (Patient Questionnaire) Total Score',
       'MDS-UPDRS Part II Total Score', 'MDS-UPDRS Part III Total Score',
       'MDS-UPDRS Part IV Total Score', 'Height (cm)', 'Weight (kg)',
       'Temperature (Celsius)', 'Supine BP - systolic (mmHg)',
       'Supine BP - diastolic (mmHg)', 'Supine heart rate (bpm)',
       'Standing BP - systolic (mmHg)', 'Standing BP - diastolic (mmHg)',
       'Standing heart rate (bpm)', 'Sporadic PD at Enrollment',
       'RBD at Enrollment', 'Pink1 Mutation at Enrollment',
       'Parkin Mutation at Enrollment', 'LRRK2 Mutation at Enrollment',
       'SNCA Mutation at Enrollment', 'GBA Mutation at Enrollment',
       'Birth Date', 'Sex of participant at birth', 'Identify self as Asian',
       'Identify self as Black/African American',
       'Identify self as Hawaiian/Other Pacific Islander',
       'Identify self as American Indian/Alaska Native',
       'Identify self as White',
       'Number of ye

# Motor Data Model

In [28]:
motor_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
motor_data.drop(columns=['Unnamed: 0'], inplace=True)
motor_data['PATNO']=motor_data['PATNO'].astype(str)




# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation=col_assignation.loc[col_assignation['df_main'].isin(['MOTOR']),:]
col_assignation=['PATNO','Visit ID'] + col_assignation['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

motor_data = motor_data.loc[:, motor_data.columns.isin(col_assignation)]
motor_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Motor_1Year_12SEP2025.csv')
motor_data

,PATNO,Visit ID,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Is the participant on medication or receiving deep brain stimulation for treating the symptoms of Parkinsons disease?,Is subject on medication for PD,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,...,DBS_Post_Transition,SCHWAB & ENGLAND ADL,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,8,9.0,1,1,0,17,2,2.0,...,0.0,95,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,2,7.0,1,1,0,23,2,0.0,...,0.0,95,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,6,5.0,1,1,0,14,1,0.0,...,0.0,95,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,7,17.0,1,1,0,17,2,3.0,...,0.0,90,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,6,5.0,1,1,0,14,2,2.0,...,0.0,95,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,14,12.0,1,1,0,20,2,3.0,...,0.0,90,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,5,8.0,1,1,0,16,2,5.0,...,0.0,95,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,16,12.0,1,1,1,32,2,3.0,...,0.0,90,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,11,7.0,1,1,0,17,2,5.0,...,0.0,100,44.0,27,40.0,4.0,-1,0.0,10,-5.0


# COGNITIVE DATA MODEL

In [29]:
cog_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
cog_data.drop(columns=['Unnamed: 0'], inplace=True)
cog_data['PATNO']=cog_data['PATNO'].astype(str)




# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation1=col_assignation.loc[col_assignation['df_main'].isin(['NON_MOTOR']),:]

col_assignation1=col_assignation1.loc[~col_assignation1['df_secundario'].isin(['EPWORTH','REMSLEEP']),:]

col_assignation2=col_assignation.loc[col_assignation['df_main'].isin(['MEDICAL']) & col_assignation['df_secundario'].isin(['NEURO']),:]
col_assignation_final=['PATNO','Visit ID'] + col_assignation1['df_secundario_col'].tolist() + col_assignation2['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

cog_data = cog_data.loc[:, cog_data.columns.isin(col_assignation_final)]
cog_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Cognitive_1Year_12SEP2025.csv')
cog_data


,PATNO,Visit ID,Letter-Number Sequencing Scaled Score,STAI PART I Score,STAI PART II Score,STAI Total Score,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,...,CN II-XII assessment,Benton Judgement of Line Orientation Scaled Score,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,11,48.0,38.0,86.0,8,9.0,17,2.0,...,Normal,13.49,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,12,49.0,47.0,96.0,2,7.0,23,0.0,...,Normal,11.47,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,12,49.0,49.0,98.0,6,5.0,14,0.0,...,Normal,12.80,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,12,44.0,41.0,85.0,7,17.0,17,3.0,...,Normal,15.00,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,8,48.0,45.0,93.0,6,5.0,14,2.0,...,Normal,6.66,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,10,47.0,44.0,91.0,14,12.0,20,3.0,...,Normal,10.83,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,12,46.0,41.0,87.0,5,8.0,16,5.0,...,Normal,10.14,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,9,57.0,47.0,104.0,16,12.0,32,3.0,...,Normal,12.34,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,11,53.0,43.0,96.0,11,7.0,17,5.0,...,Normal,11.06,44.0,27,40.0,4.0,-1,0.0,10,-5.0


# SLEEP DATA

In [30]:
sleep_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
sleep_data.drop(columns=['Unnamed: 0'], inplace=True)
sleep_data['PATNO']=sleep_data['PATNO'].astype(str)




# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation1=col_assignation.loc[col_assignation['df_main'].isin(['NON_MOTOR']),:]

col_assignation1=col_assignation1.loc[col_assignation1['df_secundario'].isin(['EPWORTH','REMSLEEP']),:]
col_assignation_final=['PATNO','Visit ID'] + col_assignation1['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

sleep_data = sleep_data.loc[:, sleep_data.columns.isin(col_assignation_final)]
sleep_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Sleep_1Year_12SEP2025.csv')
sleep_data

,PATNO,Visit ID,Sitting and reading,Watching TV,"Sitting, inactive in a public place",As a passenger in a car for an hour,Lying down to rest in the afternoon,Sitting and talking to someone,Sitting quietly after lunch,"In a car, while stopped in traffic",...,Epilepsy,Inflammatory disease of the brain,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,0,1,0.0,1,2,0.0,0,0,...,0,0,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,1,0,0.0,0,1,0.0,0,0,...,0,0,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,1,1,0.0,0,1,0.0,1,0,...,0,0,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,0,0,0.0,1,1,0.0,0,0,...,0,0,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,1,1,0.0,1,1,0.0,0,0,...,0,0,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,2,2,0.0,1,3,0.0,1,0,...,0,0,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,0,2,0.0,0,2,0.0,0,0,...,0,0,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,3,3,2.0,1,2,0.0,2,0,...,0,0,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,3,3,2.0,3,3,2.0,3,2,...,0,0,44.0,27,40.0,4.0,-1,0.0,10,-5.0


# MEDICATION MODEL

In [31]:
med_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
med_data.drop(columns=['Unnamed: 0'], inplace=True)
med_data['PATNO']=med_data['PATNO'].astype(str)




# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation=col_assignation.loc[col_assignation['df_main'].isin(['MEDICAL']) & col_assignation['df_secundario'].isin(['LEDD_MEDS','NOT_PD_MEDS']),:]
col_assignation=['PATNO','Visit ID'] + col_assignation['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

med_data = med_data.loc[:, med_data.columns.isin(col_assignation)]
med_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Medication_1Year_12SEP2025.csv')
med_data

,PATNO,Visit ID,Month_11_Medication,Month_11_Dose_Strength_mg_amantadine,Month_11_Total_mg_per_day_amantadine,Month_11_Dose_Strength_mg_selegiline,Month_11_Total_mg_per_day_selegiline,Month_11_Dose_Strength_mg_rotigotine,Month_11_Total_mg_per_day_rotigotine,Month_11_Dose_Strength_mg_pramipexole,...,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,['carbidopa levodopa'],0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17,2.0,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,['carbidopa levodopa'],0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23,0.0,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,['carbidopa levodopa'],0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14,0.0,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,['carbidopa levodopa'],0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17,3.0,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,['carbidopa levodopa'],0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14,2.0,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,"['amantadine', 'carbidopa levodopa']",100.0,200.0,0.0,0.0,0.0,0.0,0.0,...,20,3.0,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,"['rasagiline', 'carbidopa levodopa']",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16,5.0,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,"['ropinirole', 'carbidopa levodopa']",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,32,3.0,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,"['pramipexole', 'amantadine', 'carbidopa levod...",100.0,200.0,0.0,0.0,0.0,0.0,0.0,...,17,5.0,44.0,27,40.0,4.0,-1,0.0,10,-5.0


In [32]:
med_data.columns

Index(['PATNO', 'Visit ID', 'Month_11_Medication',
       'Month_11_Dose_Strength_mg_amantadine',
       'Month_11_Total_mg_per_day_amantadine',
       'Month_11_Dose_Strength_mg_selegiline',
       'Month_11_Total_mg_per_day_selegiline',
       'Month_11_Dose_Strength_mg_rotigotine',
       'Month_11_Total_mg_per_day_rotigotine',
       'Month_11_Dose_Strength_mg_pramipexole',
       ...
       'MDS-UPDRS Part III Total Score', 'MDS-UPDRS Part IV Total Score',
       'MDS-UPDRS Total Score 1YearProg', 'UPDRS_III_1YearProg',
       'MDS-UPDRS Total Score', 'Δ TOTAL_UPDRS_1Y', 'Δ UPDRS_I_1Y',
       'Δ UPDRS_II_1Y', 'Δ UPDRS_III_1Y', 'Δ UPDRS_IV_1Y'],
      dtype='object', length=397)

# AE_EVENTS

In [33]:
ae_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS1Y.csv')
ae_data.drop(columns=['Unnamed: 0'], inplace=True)
ae_data['PATNO']=ae_data['PATNO'].astype(str)

# selection of cols
col_assignation=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ASSIGNATION_COL.csv')
col_assignation=col_assignation.loc[col_assignation['df_main'].isin(['MEDICAL']) & col_assignation['df_secundario'].isin(['AE_CLINIC','AE_NO_CLINIC']),:]
col_assignation=['PATNO','Visit ID'] + col_assignation['df_secundario_col'].tolist() + ['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part IV Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Total Score','MDS-UPDRS Total Score 1YearProg','UPDRS_III_1YearProg','Δ TOTAL_UPDRS_1Y','Δ UPDRS_I_1Y','Δ UPDRS_II_1Y','Δ UPDRS_III_1Y','Δ UPDRS_IV_1Y']

ae_data = ae_data.loc[:, ae_data.columns.isin(col_assignation)]
ae_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Adverse_Events_1Year_12SEP2025.csv')
ae_data

,PATNO,Visit ID,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,Any adverse events observed?,LP performed on assessment date,Skin Biopsy performed on assessment date,Dopamine Imaging performed on assessment date,...,R_LP,R_BiopsySkin,MDS-UPDRS Total Score 1YearProg,UPDRS_III_1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,8,9.0,17,2.0,No,Unchecked,Unchecked,Unchecked,...,0,0,34.0,9,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,2,7.0,23,0.0,No,Unchecked,Unchecked,Unchecked,...,0,0,19.0,16,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,6,5.0,14,0.0,No,Unchecked,Unchecked,Unchecked,...,0,0,33.0,17,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,7,17.0,17,3.0,No,Unchecked,Unchecked,Unchecked,...,0,0,59.0,31,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,6,5.0,14,2.0,No,Unchecked,Unchecked,Unchecked,...,0,0,23.0,11,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,14,12.0,20,3.0,No,Unchecked,Unchecked,Unchecked,...,1,0,32.0,17,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,5,8.0,16,5.0,No,Unchecked,Unchecked,Unchecked,...,0,0,34.0,19,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,16,12.0,32,3.0,No,Unchecked,Unchecked,Unchecked,...,0,0,66.0,24,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,11,7.0,17,5.0,No,Unchecked,Unchecked,Unchecked,...,0,0,44.0,27,40.0,4.0,-1,0.0,10,-5.0


# Unin de Datos 2Y

In [34]:


V04_V08_data=pd.merge(V04_data,V08_data[['PATNO','UPDRS_V08']], how='inner', on='PATNO')
V04_V08_data['Visit ID']='V04_V08'
print(f'V04_V08_data shape after merge: {V04_V08_data.shape}')
V04_V08_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS2Y_V04_V08_data.csv')

V06_V10_data=pd.merge(V06_data,V10_data[['PATNO','UPDRS_V10']], how='inner', on='PATNO')
V06_V10_data['Visit ID']='V06_V10'
print(f'V06_V10_data shape after merge: {V06_V10_data.shape}')
V06_V10_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS2Y_V06_V10_data.csv')

V08_V12_data=pd.merge(V08_data,V12_data[['PATNO','UPDRS_V12']], how='inner', on='PATNO')
V08_V12_data['Visit ID']='V08_V12'
print(f'V08_V12_data shape after merge: {V08_V12_data.shape}')
V08_V12_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/Presence_matrix/GS2Y_V08_V12_data.csv')

V04_V08_data shape after merge: (201, 22)
V06_V10_data shape after merge: (170, 22)
V08_V12_data shape after merge: (161, 22)


In [35]:
def data_loader2(df,id1,id2):

    list_data = df.columns.tolist()
    list_id1 = [w.replace("_"+id1, "") for w in list_data if w.endswith("_"+id1)]
    list_id2 = [w.replace("_"+id2, "") for w in list_data if w.endswith("_"+id2)]

    dict_data={id1: list_id1, id2: list_id2}
    ids = df['PATNO'].astype(str).tolist()
    visit = [id1 + "_" + id2] * len(ids)
    new_df=pd.DataFrame({'PATNO': ids,'Visit ID': visit})
    for key in dict_data:
        if key==id1:
            for col in dict_data[key]:
                df_specific = pd.read_csv(DATA_PATH_DICT[col])
                df_specific['PATNO']=df_specific['PATNO'].astype(str)
                df_specific = df_specific.loc[(df_specific['PATNO'].isin(ids)) & (df_specific['Visit ID']==id1), :]
                df_specific.drop(columns=['Visit ID'], inplace=True)
                new_df = pd.merge(new_df, df_specific, how='left', on='PATNO')

        if key==id2:
            for col in dict_data[key]:
                df_specific = pd.read_csv(DATA_PATH_DICT[col])
                df_specific['PATNO']=df_specific['PATNO'].astype(str)
                df_specific = df_specific.loc[(df_specific['PATNO'].isin(ids)) & (df_specific['Visit ID']==id2), :]
                df_specific['MDS-UPDRS Total Score 1YearProg']=df_specific[['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score']].astype(float).sum(axis=1)
                df_specific['UPDRS_I_1YearProg']=df_specific['MDS-UPDRS Part I (Patient Questionnaire) Total Score']
                df_specific['UPDRS_II_1YearProg']=df_specific['MDS-UPDRS Part II Total Score']
                df_specific['UPDRS_III_1YearProg']=df_specific['MDS-UPDRS Part III Total Score']
                df_specific['UPDRS_IV_1YearProg']=df_specific['MDS-UPDRS Part IV Total Score']
                df_specific= df_specific[['PATNO','MDS-UPDRS Total Score 1YearProg','UPDRS_I_1YearProg','UPDRS_II_1YearProg','UPDRS_III_1YearProg','UPDRS_IV_1YearProg']]
                new_df = pd.merge(new_df, df_specific, how='left', on='PATNO')

    new_df['MDS-UPDRS Total Score']=new_df[['MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score','MDS-UPDRS Part IV Total Score']].astype(float).sum(axis=1)
    new_df['Δ TOTAL_UPDRS_2Y']=new_df['MDS-UPDRS Total Score 1YearProg'] - new_df['MDS-UPDRS Total Score']
    new_df['Δ UPDRS_I_2Y']=new_df['UPDRS_I_1YearProg'] - new_df['MDS-UPDRS Part I (Patient Questionnaire) Total Score']
    new_df['Δ UPDRS_II_2Y']=new_df['UPDRS_II_1YearProg'] - new_df['MDS-UPDRS Part II Total Score']
    new_df['Δ UPDRS_III_2Y']=new_df['UPDRS_III_1YearProg'] - new_df['MDS-UPDRS Part III Total Score']
    new_df['Δ UPDRS_IV_2Y']=new_df['UPDRS_IV_1YearProg'] - new_df['MDS-UPDRS Part IV Total Score']
    return new_df


# visita progression anual


V04_V08_final=data_loader2(V04_V08_data,'V04','V08')
print(f'V04_V08_final shape:{ V04_V08_final.shape}')
V04_V08_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS2Y_data_V04_V08_final.csv')


V06_V10_final=data_loader2(V06_V10_data,'V06','V10')
print(f'V06_V10_final shape:{ V06_V10_final.shape}')
V06_V10_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS2Y_data_V06_V10_final.csv')


V08_V12_final=data_loader2(V08_V12_data,'V08','V12')
print(f'V08_V12_final shape:{ V08_V12_final.shape}')
V08_V12_final.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/GS2Y_data_V08_V12_final.csv')


# data final concatenada
FINAL_DATA_2=pd.concat([V04_V08_final,V06_V10_final,V08_V12_final], axis=0)
print(f'FINAL_DATA shape:{ FINAL_DATA_2.shape}')
FINAL_DATA_2.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Final_GS_DATA/DATA/FINAL_DATA_GS2Y.csv')


V04_V08_final shape:(201, 501)
V06_V10_final shape:(170, 501)
V08_V12_final shape:(161, 501)
FINAL_DATA shape:(532, 501)
